In [1]:
%cd ..

/Users/stanf/Documents/Uni/Project_AI/two-tower-confounding


In [2]:
import altair as alt
import pandas as pd

from notebooks.utils import load
from two_tower_confounding.simulation.simulator import get_position_bias

In [3]:
def theme():
    return {
        "config": {
             "title": {
                "font": "serif",
                "fontWeight": "normal",
                "fontSize": 16,
                "dx": 5,
                 "subtitleFont": "serif",
                 "subtitleFontSize": 14,
            },
            "axis": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 14
            },
            "headerColumn": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16
            },
            "text": {
                "font": "serif",
                "fontSize": 14,
            },
            "legend": {
                "symbolOpacity": 1,
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16,
            }
        },
    }

alt.themes.register("latex", theme)
alt.themes.enable("latex")

/var/folders/nb/j_h4xjxd1tl7g7169hm8cdnc0000gn/T/ipykernel_96520/782434750.py:44: AltairDeprecationWarning: 
Deprecated since `altair=5.5.0`. Use altair.theme instead.
Most cases require only the following change:

    # Deprecated
    alt.themes.enable('quartz')

    # Updated
    alt.theme.enable('quartz')

If your code registers a theme, make the following change:

    # Deprecated
    def custom_theme():
        return {'height': 400, 'width': 700}
    alt.themes.register('theme_name', custom_theme)
    alt.themes.enable('theme_name')

    # Updated
    @alt.theme.register('theme_name', enable=True)
    def custom_theme():
        return alt.theme.ThemeConfig(
            {'height': 400, 'width': 700}
        )

See the updated User Guide for further details:
    https://altair-viz.github.io/user_guide/api.html#theme
    https://altair-viz.github.io/user_guide/customization.html#chart-themes
  alt.themes.register("latex", theme)


ThemeRegistry.enable('latex')

In [6]:
def plot_bias(experiment, policy_strength, title="", subtitle="", width=225, height=125, color="blues", show_axis_title=True):
    scheme = {
        "blue": ["#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#636363"],
        "green": ["#C7E9C0", "#A1D99B", "#74C476", "#31A354", "#636363"],
        "orange": ["#FDD0A2", "#FD8D3C", "#FD8D3C", "#E6550E", "#636363"],
    }[color]

    
    bias_df = load(experiment, "bias.csv")
    bias_df = bias_df[bias_df["policy_strength"] == policy_strength]

    
    n_positions = bias_df.position.max() + 1
    true_bias_df = pd.DataFrame(
        {
            "position": range(n_positions),
            "examination": get_position_bias(n_positions, 1),
            "policy_temperature": "Simulated Bias"
        }
    )

    bias_df = pd.concat([bias_df, true_bias_df])
    bias_df["position"] += 1

    base = alt.Chart(bias_df, width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)).encode(
        x=alt.X("position:Q", title="Position").axis(labelAngle=0, values=[1, 5, 10, 15, 20, 25]).scale(nice=False)
    )
    
    line = base.mark_line(point=True, opacity=1).encode(
        y=alt.Y("mean(examination):Q", title="Bias Logits" if show_axis_title else None),
        color=alt.Color("policy_temperature:O", title="Temperature 𝜏").scale(range=scheme)
    )
    
    ci = base.mark_errorband(extent="ci").encode(
        y=alt.Y("examination:Q", title="Bias Logits" if show_axis_title else None),
        color=alt.Color("policy_temperature:O").scale(range=scheme)
    )
    
    return ci + line





linear = plot_bias("test", policy_strength=1, color="green", title="Linear Relevance Tower", subtitle="linear rel., policy strength 𝛼 = 1")
deep = plot_bias("2-model-fit-deep", policy_strength=1, color="green", title="Deep Relevance Tower", subtitle="non-linear rel., policy strength 𝛼 = 1", show_axis_title=False)
embedding = plot_bias("2-model-fit-embedding", policy_strength=1, color="green", title="Embedding Relevance Tower", subtitle="expert rel., policy strength 𝛼 = 1", show_axis_title=False)


chart = (
    linear | deep | embedding
).configure_line(
    size=2
).configure_point(
    size=15
).resolve_scale(
    y="shared"
).configure_concat(
    spacing=5
)

chart

alt.HConcatChart(...)

In [5]:
def plot_relevance(experiment, policy_strength, title="", subtitle="", width=225, height=125, color="blues", show_axis_title=True):
    scheme = {
        "blue": ["#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#636363"],
        "green": ["#C7E9C0", "#A1D99B", "#74C476", "#31A354", "#636363"],
        "orange": ["#FDD0A2", "#FD8D3C", "#FD8D3C", "#E6550E", "#636363"],
    }[color]

    df = load(experiment, "test.csv")

    base = alt.Chart(df, width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)).encode(
        x=alt.X("policy_strength:O", title="Policy Strength 𝛼").axis(labelAngle=0)
    )
    
    line = base.mark_line(point=True, opacity=1).encode(
        y=alt.Y("mean(ndcg@10):Q", title="nDCG@10" if show_axis_title else None).scale(zero=False),
        color=alt.Color("policy_temperature:O", title="Temperature 𝜏").scale(range=scheme)
    )
    
    ci = base.mark_errorband(extent="ci").encode(
        y=alt.Y("ndcg@10:Q", title="nDCG@10" if show_axis_title else None).scale(zero=False),
        color=alt.Color("policy_temperature:O").scale(range=scheme)
    )

    return ci + line





linear = plot_relevance("2-model-fit-linear", policy_strength=1, color="green", title="Linear Relevance Tower", subtitle="linear rel.")
deep = plot_relevance("2-model-fit-deep", policy_strength=1, color="green", title="Deep Relevance Tower", subtitle="non-linear rel.", show_axis_title=False)
embedding = plot_relevance("2-model-fit-embedding", policy_strength=1, color="green", title="Embedding Relevance Tower", subtitle="expert rel.", show_axis_title=False)


chart = (
    linear | deep | embedding
).configure_line(
    size=2
).configure_point(
    size=15
).resolve_scale(
    y="shared"
).configure_concat(
    spacing=5
)

chart

alt.HConcatChart(...)